# 其他语言 OpenAI Whisper-large-v2 训练

## 其他语言统一配置

In [1]:
from pathlib import Path

LANG = "fr"  # 切换为 "de" 或 "nn"

LANG_SPECS = {
    "fr": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_fr"
            "/cv-corpus-27.0-2026-09-11/fr"
        ),
        "language": "french",
        "abbr": "fr",
    },
    "de": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_ge"
            "cv-corpus-27.0-2026-09-11/de"
        ),
        "language": "german",
        "abbr": "de",
    },
    "nn": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_nork"
            "/cv-corpus-27.0-2026-09-11/nn-NO"
        ),
        "language": "nynorsk",
        "abbr": "nn",
    },
}

CFG = LANG_SPECS[LANG]
CV_ROOT = CFG["cv_root"]
WHISPER_LANGUAGE = CFG["language"]
LANGUAGE_ABBR = CFG["abbr"]
TASK = "transcribe"

## 全局参数设置

In [2]:
model_name_or_path = "openai/whisper-large-v2"
model_dir = (
    f"/sigan/ai-ml/models/"
    f"whisper-large-v2-{LANGUAGE_ABBR}-lora"
)

language = WHISPER_LANGUAGE
language_abbr = LANGUAGE_ABBR
task = TASK
dataset_name = "mozilla-foundation/common_voice_11_0"

# 全量训练使用较小的值
# batch_size=64
batch_size = 2

## 数据准备

### 下载数据集 Common Voice

Common Voice 11.0 数据集包含许多不同语言的录音，总时长达数小时。

本教程以中文数据为例，展示如何使用 LoRA 在 Whisper-large-v2 上进行微调训练。

首先，初始化一个DatasetDict结构，并将训练集（将训练+验证拆分为训练集）和测试集拆分好，按照中文数据集构建配置加载到内存中：

### Common Voice已从HuggingFace移除，使用备份非官方版本

In [3]:
from pathlib import Path

from datasets import Audio, DatasetDict, load_dataset

dataset_path = CV_ROOT
clips_path = dataset_path / "clips"


def load_split(tsv_name):
    # 读取 TSV
    dataset = load_dataset(
        "csv",
        data_files=str(dataset_path / tsv_name),
        delimiter="\t",
        keep_default_na=False,
    )["train"]

    # path 只是文件名，这里补成完整音频路径
    dataset = dataset.map(
        lambda batch: {
            "audio": [
                str(clips_path / name)
                for name in batch["path"]
            ]
        },
        batched=True,
    )

    # 统一转换成 16kHz 音频
    dataset = dataset.cast_column(
        "audio",
        Audio(sampling_rate=16000),
    )

    # 后面只用 audio 和 sentence
    return dataset.select_columns(
        ["audio", "sentence"]
    )


common_voice = DatasetDict({
    "train": load_split("train.tsv"),
    "validation": load_split("dev.tsv"),
})

print(common_voice)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/621982 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/16207 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 621982
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 16207
    })
})


## 预处理训练数据集


In [4]:
from transformers import AutoFeatureExtractor, AutoTokenizer, AutoProcessor

# 从预训练模型加载特征提取器
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name_or_path)

# 从预训练模型加载分词器，可以指定语言和任务以获得最适合特定需求的分词器配置
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, language=language, task=task)

# 从预训练模型加载处理器，处理器通常结合了特征提取器和分词器，为特定任务提供一站式的数据预处理
processor = AutoProcessor.from_pretrained(model_name_or_path, language=language, task=task)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


#### 降采样音频数据

查看`common_voice` 数据集介绍，你会发现其音频是以48kHz的采样率进行采样的.

而`Whisper`模型是在16kHZ的音频输入上预训练的，因此我们需要将音频输入降采样以匹配模型预训练时使用的采样率。

通过在音频列上使用`cast_column`方法，并将`sampling_rate`设置为16kHz来对音频进行降采样。

下次调用时，音频输入将实时重新取样：

In [5]:
from datasets import Audio

common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

### 整合以上数据处理为一个函数

该数据预处理函数应该包括：
- 通过加载音频列将音频输入重新采样为16kHZ。
- 使用特征提取器从音频数组计算输入特征。
- 将句子列标记化为输入标签。

In [6]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

### 数据抽样（演示需要）

在 Whisper-Large-v2 上使用小规模数据进行演示训练，保持以下训练参数不变（batch_size=64）。

使用 640 个样本训练，320个样本验证和评估，恰好使得1个 epoch 仅需10 steps 即可完成训练。

（在 NVIDIA T4 上需要10-15分钟）

### 如果全量训练，则使用完整数据代替抽样

#### 在 common_voice.map() 前限制训练数据

法语和德语分别有约 62 万条训练数据，不能直接全部做 Mel 特征预处理

In [7]:
MAX_TRAIN_SAMPLES = 20000

common_voice["train"] = (
    common_voice["train"]
    .shuffle(seed=42)
    .select(
        range(
            min(
                MAX_TRAIN_SAMPLES,
                len(common_voice["train"]),
            )
        )
    )
)

In [8]:
# 完整数据训练，尝试开启 `num_proc=8` 参数多进程并行处理（如阻塞无法运行，则不使用此参数）
tokenized_common_voice = common_voice.map(prepare_dataset, num_proc=2)

Map (num_proc=2):   0%|          | 0/20000 [00:00<?, ? examples/s]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Map (num_proc=2):   0%|          | 0/16207 [00:00<?, ? examples/s]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [9]:
tokenized_common_voice

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence', 'input_features', 'labels'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['audio', 'sentence', 'input_features', 'labels'],
        num_rows: 16207
    })
})

### 自定义语音数据整理器

定义了一个针对语音到文本（Seq2Seq）模型的自定义数据整理器类，特别适用于输入为语音特征、输出为文本序列的数据集。


这个整理器（`DataCollatorSpeechSeq2SeqWithPadding`）旨在将数据点批量打包，将每个批次中的`attention_mask`填充到最大长度，以保持批处理中张量形状的一致性，并用`-100`替换填充值，以便在损失函数中被忽略。这对于神经网络的高效训练至关重要。

In [10]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

# 定义一个针对语音到文本任务的数据整理器类
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any  # 处理器结合了特征提取器和分词器

    # 整理器函数，将特征列表处理成一个批次
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 从特征列表中提取输入特征，并填充以使它们具有相同的形状
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 从特征列表中提取标签特征（文本令牌），并进行填充
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 使用-100替换标签中的填充区域，-100通常用于在损失计算中忽略填充令牌
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 如果批次中的所有序列都以句子开始令牌开头，则移除它
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        # 将处理过的标签添加到批次中
        batch["labels"] = labels

        return batch  # 返回最终的批次，准备好进行训练或评估

In [11]:
# 用给定的处理器实例化数据整理器
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 模型准备

### 加载预训练模型（int8 精度）

使用 `int8 ` 精度加载预训练模型，进一步降低显存需求。

In [12]:
from transformers import AutoModelForSpeechSeq2Seq

model = AutoModelForSpeechSeq2Seq.from_pretrained(model_name_or_path, load_in_8bit=True, device_map="auto")

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
# 设置模型配置中的forced_decoder_ids属性为None
model.config.forced_decoder_ids = None  # 这通常用于指定在解码（生成文本）过程中必须使用的特定token的ID，设置为None表示没有这样的强制要求

# 设置模型配置中的suppress_tokens列表为空
model.config.suppress_tokens = []  # 这用于指定在生成过程中应被抑制（不生成）的token的列表，设置为空列表表示没有要抑制的token

### PEFT 微调前的模型处理

在使用 `peft` 训练 int8 模型之前，需要进行一些预处理：
- 将所有非 `int8` 精度模块转换为全精度（`fp32`）以保证稳定性
- 为输入嵌入层添加一个 `forward_hook`，以启用输入隐藏状态的梯度计算
- 启用梯度检查点以实现更高效的内存训练

使用 `peft` 库预定义的工具函数 `prepare_model_for_int8_training`，便可自动完成以上模型处理工作。

In [14]:
from peft import prepare_model_for_int8_training

model = prepare_model_for_int8_training(model)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/peft/utils/other.py:141: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(


### LoRA Adapter 配置

在 `peft` 中使用`LoRA`非常简捷，借助 `PeftModel`抽象，我们可以快速使用低秩适配器（LoRA）到任意模型。

通过使用 `peft` 中的 `get_peft_model` 工具函数来实现。

#### 关于 LoRA 超参数的说明：
```
MatMul(B,A) * Scaling
Scaling = LoRA_Alpha / Rank
```

In [15]:
from peft import LoraConfig, PeftModel, LoraModel, LoraConfig, get_peft_model

# 创建一个LoraConfig对象，用于设置LoRA（Low-Rank Adaptation）的配置参数
# config = LoraConfig(
#     r=4,  # LoRA的秩，影响LoRA矩阵的大小
#     lora_alpha=64,  # LoRA适应的比例因子
#     # 指定将LoRA应用到的模型模块，通常是attention和全连接层的投影。
#     target_modules=["q_proj", "v_proj"],
#     lora_dropout=0.05,  # 在LoRA模块中使用的dropout率
#     bias="none",  # 设置bias的使用方式，这里没有使用bias
# )
config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

### 使用get_peft_model函数和给定的配置来获取一个PEFT模型

In [16]:
peft_model = get_peft_model(model, config)

In [17]:
# 打印 LoRA 微调训练的模型参数
peft_model.print_trainable_parameters()

trainable params: 3,932,160 || all params: 1,547,237,120 || trainable%: 0.25414074863974306


## 模型训练

#### Seq2SeqTrainingArguments 训练参数

**关于设置训练步数和评估步数**

基于 epochs 设置：

```python
    num_train_epochs=3,  # 训练的总轮数
    evaluation_strategy="epoch",  # 设置评估策略，这里是在每个epoch结束时进行评估
    warmup_steps=50,  # 在训练初期增加学习率的步数，有助于稳定训练
```

基于 steps 设置：

```python
    max_steps=100, # 训练总步数
    evaluation_strategy="steps", 
    eval_steps=25, # 评估步数
```

In [18]:
from transformers import Seq2SeqTrainingArguments

# 设置序列到序列模型训练的参数
training_args = Seq2SeqTrainingArguments(
    output_dir=model_dir,  # 指定模型输出和保存的目录
    per_device_train_batch_size=batch_size,  # 每个设备上的训练批量大小
    per_device_eval_batch_size=batch_size,  # 每个设备上的评估批量大小

    gradient_accumulation_steps=8,
    
    # learning_rate=1e-3,  # 学习率
    learning_rate=1e-4,  # 降低学习率，全数据微调更稳定
    
    num_train_epochs=1,  # 训练的总轮数
    # evaluation_strategy="epoch",  # 设置评估策略，这里是在每个epoch结束时进行评估
    evaluation_strategy="steps",  # 设置评估策略，全量数据训练每 500 步记录一次验证 loss
    eval_steps=500,
    
    warmup_steps=50,  # 在训练初期增加学习率的步数，有助于稳定训练
    fp16=True,  # 启用混合精度训练，可以提高训练速度，同时减少内存使用
    generation_max_length=128,  # 生成任务的最大长度
    
    # logging_steps=10,  # 指定日志记录的步骤，用于跟踪训练进度
    logging_steps=50,  # 全量数据微调放大点
    
    remove_unused_columns=False,  # 是否删除不使用的列，以减少数据处理开销
    label_names=["labels"],  # 指定标签列的名称，用于训练过程中
    
    # 每 500 步保存 checkpoint
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    # 只比较 loss，不执行昂贵的文本生成
    predict_with_generate=False,

    # 按最低验证 loss 保留最佳 checkpoint
    metric_for_best_model="loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

### 实例化 Seq2SeqTrainer 训练器

In [19]:
# 全数据微调
from transformers import Seq2SeqTrainer

# 完整验证集较大，先固定抽取 1000 条观察验证 loss
eval_dataset = (
    tokenized_common_voice["validation"]
    .shuffle(seed=42)
    .select(
        range(
            min(
                300,
                len(tokenized_common_voice["validation"]),
            )
        )
    )
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=peft_model,
    train_dataset=tokenized_common_voice["train"],
    eval_dataset=eval_dataset,
    data_collator=data_collator,

    # 这里是文本 tokenizer，不是 feature_extractor
    tokenizer=processor.tokenizer,
)

# 训练时关闭 KV Cache
peft_model.config.use_cache = False

In [20]:
trainer.train()
# 中断后继续训练
# trainer.train(resume_from_checkpoint=True)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_gr

Step,Training Loss,Validation Loss
500,0.292800,0.261107
1000,0.249700,0.246878


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/sigan/miniconda3/envs/pe

TrainOutput(global_step=1250, training_loss=0.49958243942260744, metrics={'train_runtime': 8041.7219, 'train_samples_per_second': 2.487, 'train_steps_per_second': 0.155, 'total_flos': 4.2576666624e+19, 'train_loss': 0.49958243942260744, 'epoch': 1.0})

### 训练后查看 loss 变化

In [21]:
for item in trainer.state.log_history:
    if "loss" in item:
        print(
            "step:", item.get("step"),
            "train_loss:", item["loss"],
        )

    if "eval_loss" in item:
        print(
            "step:", item.get("step"),
            "eval_loss:", item["eval_loss"],
        )

step: 50 train_loss: 3.1449
step: 100 train_loss: 1.7633
step: 150 train_loss: 1.0772
step: 200 train_loss: 0.8109
step: 250 train_loss: 0.4351
step: 300 train_loss: 0.276
step: 350 train_loss: 0.2773
step: 400 train_loss: 0.2871
step: 450 train_loss: 0.2557
step: 500 train_loss: 0.2928
step: 500 eval_loss: 0.2611073851585388
step: 550 train_loss: 0.2678
step: 600 train_loss: 0.2609
step: 650 train_loss: 0.2709
step: 700 train_loss: 0.2573
step: 750 train_loss: 0.2644
step: 800 train_loss: 0.2535
step: 850 train_loss: 0.2698
step: 900 train_loss: 0.2553
step: 950 train_loss: 0.2465
step: 1000 train_loss: 0.2497
step: 1000 eval_loss: 0.24687805771827698
step: 1050 train_loss: 0.2527
step: 1100 train_loss: 0.2658
step: 1150 train_loss: 0.2617
step: 1200 train_loss: 0.2273
step: 1250 train_loss: 0.2658


### 保存 LoRA 模型(Adapter)

In [22]:
trainer.save_model(model_dir)

In [ ]:
peft_model.eval()